# Proposed B-TGPRF Model and Validation-Only Architecture Selection

This notebook computes the proposed **B-TGPRF** (*Bayesian Temporal Graph Probabilistic Risk Forecaster*) as one coherent architecture. Candidate component choices are selected only from validation data. Test and external-test partitions are evaluated only after the configuration is fixed. The notebook saves computation artifacts for the final manuscript-results notebook.

In [1]:

import os, time, json, math, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from IPython.display import display
warnings.filterwarnings("ignore")

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import BayesianRidge, LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.isotonic import IsotonicRegression
from sklearn.ensemble import HistGradientBoostingRegressor
try:
    from scipy.optimize import minimize
    SCIPY_OPTIMIZE_AVAILABLE = True
except Exception:
    SCIPY_OPTIMIZE_AVAILABLE = False
import joblib

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_UNIFIED = PROJECT_ROOT / "Data" / "Unified"
TABLES = PROJECT_ROOT / "Tables"
MODELS = PROJECT_ROOT / "Models"
REPORTS = PROJECT_ROOT / "Reports"
ARTIFACT_DIR = MODELS / "proposed_model_artifacts"
for p in [TABLES, MODELS, REPORTS, ARTIFACT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print("Execution environment: ai-classic-ml-cpu")
print(f"Random seed: {RANDOM_SEED}")
print(f"SciPy optimizer available: {SCIPY_OPTIMIZE_AVAILABLE}")


Project root: /Users/talgatazykanov/Desktop/Science works/Bayiessian_Asem_Almaty_2026_05/Bayesian_PowerGrid_Risk_Forecasting_N1
Execution environment: ai-classic-ml-cpu
Random seed: 42
SciPy optimizer available: True


## Data and Candidate Outputs

The proposed model uses the unified dataset metadata and the trained candidate predictions saved by earlier notebooks. Candidate selection is based on validation data only.

In [2]:

feature_dictionary = pd.read_csv(DATA_UNIFIED / "feature_dictionary.csv")
needed_columns = [
    "row_id", "split", "time_index", "line_id", "line_flow", "absolute_line_flow",
    "future_line_flow_h1", "future_absolute_line_flow_h1", "risk_binary_h1", "risk_label_h1",
    "future_line_flow_h6", "future_absolute_line_flow_h6", "risk_binary_h6", "risk_label_h6",
    "future_line_flow_h24", "future_absolute_line_flow_h24", "risk_binary_h24", "risk_label_h24",
    "abs_flow_q90_train", "abs_flow_q95_train", "abs_flow_q99_train", "risk_proximity_score",
    "line_centrality", "from_bus_degree", "to_bus_degree", "capacity_overload_ratio",
    "neighborhood_mean_flow", "neighborhood_max_flow", "graph_smoothed_flow", "adjacent_line_risk_count",
    "high_load_regime", "high_generation_regime", "high_ramp_regime", "low_margin_regime", "stress_regime",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "week_sin", "week_cos",
    "total_load", "mean_load", "std_load", "load_ramp", "rolling_load_mean_24", "rolling_load_std_24",
    "total_generation", "generation_ramp", "generation_volatility_24",
    "line_flow_lag1", "line_flow_lag6", "line_flow_lag24", "rolling_abs_flow_mean_6", "rolling_abs_flow_std_6",
    "rolling_abs_flow_mean_24", "rolling_abs_flow_std_24", "line_flow_ramp_1", "line_flow_ramp_6", "local_volatility_24"
]
import pyarrow.parquet as pq
available_cols = pq.ParquetFile(DATA_UNIFIED / "unified_powergrid_modeling_dataset.parquet").schema.names
use_cols = [c for c in needed_columns if c in available_cols]
model_meta = pd.read_parquet(DATA_UNIFIED / "unified_powergrid_modeling_dataset.parquet", columns=use_cols)
model_meta = model_meta.drop_duplicates("row_id").set_index("row_id")
print(f"Loaded metadata rows: {len(model_meta):,}; columns: {len(model_meta.columns)}")

def load_baseline_predictions(split):
    name = {"validation":"baseline_validation_predictions.csv", "test":"baseline_test_predictions.csv", "external_test":"baseline_external_test_predictions.csv"}[split]
    df = pd.read_csv(TABLES / name)
    out = pd.DataFrame(index=df["row_id"].drop_duplicates())
    forecast = df[df["task"].eq("forecasting")].copy()
    risk = df[df["task"].eq("classification")].copy()
    if not forecast.empty:
        out = out.join(forecast.pivot_table(index="row_id", columns="model", values="y_pred", aggfunc="first").add_prefix("fc::"), how="outer")
        out["y_true_flow"] = forecast.groupby("row_id")["y_true"].first()
        qcols = forecast[forecast["model"].eq("Quantile Gradient Boosting conformal")].set_index("row_id")[["lower_q05", "upper_q95"]]
        out = out.join(qcols.rename(columns={"lower_q05":"quantile_lower_q05", "upper_q95":"quantile_upper_q95"}), how="left")
    if not risk.empty:
        out = out.join(risk.pivot_table(index="row_id", columns="model", values="y_prob", aggfunc="first").add_prefix("prob::"), how="outer")
        out["y_true_risk"] = risk.groupby("row_id")["y_true"].first().astype(int)
    out["split"] = split
    meta_cols = [c for c in model_meta.columns if c not in out.columns]
    out = out.join(model_meta[meta_cols], how="left")
    out = out.reset_index().rename(columns={"index":"row_id"})
    return out

frames = {split: load_baseline_predictions(split) for split in ["validation", "test", "external_test"]}
summary = pd.DataFrame({k: {"rows": len(v), "forecast_candidates": len([c for c in v.columns if c.startswith("fc::")]), "risk_candidates": len([c for c in v.columns if c.startswith("prob::")]), "critical_rate": v["y_true_risk"].mean()} for k, v in frames.items()}).T.reset_index(names="split")
display(summary)


Loaded metadata rows: 3,484,800; columns: 59


,split,rows,forecast_candidates,risk_candidates,critical_rate
0,validation,59999.0,9.0,11.0,0.043617
1,test,59999.0,9.0,11.0,0.078985
2,external_test,59999.0,9.0,11.0,0.054268


## Metric Functions and B-TGPRF Components

The architecture combines a validation-selected point-forecasting backbone, graph-temporal and topology-aware features, a Bayesian posterior-style risk layer, validation calibration, residual correction, and conformal prediction intervals.

In [3]:

def expected_calibration_error(y_true, prob, n_bins=10):
    y_true = np.asarray(y_true).astype(float)
    prob = np.clip(np.asarray(prob, dtype=float), 0, 1)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (prob >= lo) & (prob < hi if hi < 1 else prob <= hi)
        if mask.any():
            ece += mask.mean() * abs(y_true[mask].mean() - prob[mask].mean())
    return float(ece)

def classification_metrics(y_true, prob, threshold):
    y_true = np.asarray(y_true).astype(int)
    prob = np.clip(np.nan_to_num(np.asarray(prob, dtype=float), nan=0.0), 0, 1)
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "Accuracy": accuracy_score(y_true, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, pred),
        "Macro-F1": f1_score(y_true, pred, average="macro", zero_division=0),
        "Weighted-F1": f1_score(y_true, pred, average="weighted", zero_division=0),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "Specificity": tn / (tn + fp + 1e-12),
        "ROC-AUC": roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else np.nan,
        "PR-AUC": average_precision_score(y_true, prob) if len(np.unique(y_true)) > 1 else np.nan,
        "False Positive Rate": fp / (fp + tn + 1e-12),
        "False Negative Rate": fn / (fn + tp + 1e-12),
        "Brier Score": brier_score_loss(y_true, prob),
        "Expected Calibration Error": expected_calibration_error(y_true, prob),
    }

def forecasting_metrics(y_true, pred, current):
    y_true = np.asarray(y_true, dtype=float)
    pred = np.asarray(pred, dtype=float)
    current = np.asarray(current, dtype=float)
    rmse = mean_squared_error(y_true, pred) ** 0.5
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": rmse,
        "sMAPE": float(np.mean(2 * np.abs(pred - y_true) / (np.abs(y_true) + np.abs(pred) + 1e-6))),
        "MAPE_safe": float(np.mean(np.abs((y_true - pred) / np.maximum(np.abs(y_true), 1e-3)))),
        "R2": r2_score(y_true, pred),
        "normalized_RMSE": rmse / (np.std(y_true) + 1e-9),
        "ramp_MAE": mean_absolute_error(y_true - current, pred - current),
    }

def interval_metrics(y_true, lo, hi):
    y_true = np.asarray(y_true, dtype=float)
    lo = np.asarray(lo, dtype=float)
    hi = np.asarray(hi, dtype=float)
    lo2 = np.minimum(lo, hi)
    hi2 = np.maximum(lo, hi)
    coverage = float(((y_true >= lo2) & (y_true <= hi2)).mean())
    pin05 = np.mean(np.maximum(0.05 * (y_true - lo2), (0.05 - 1) * (y_true - lo2)))
    pin95 = np.mean(np.maximum(0.95 * (y_true - hi2), (0.95 - 1) * (y_true - hi2)))
    return {
        "pinball_loss": float((pin05 + pin95) / 2),
        "interval_coverage_probability": coverage,
        "mean_prediction_interval_width": float(np.mean(hi2 - lo2)),
        "interval_calibration_error": abs(coverage - 0.90),
    }

def choose_threshold(y_true, prob):
    y_true = np.asarray(y_true).astype(int)
    prob = np.clip(np.asarray(prob, dtype=float), 0, 1)
    thresholds = np.unique(np.quantile(prob, np.linspace(0.02, 0.98, 120)))
    if len(thresholds) < 3:
        thresholds = np.linspace(0.02, 0.80, 80)
    scores = [f1_score(y_true, (prob >= t).astype(int), average="macro", zero_division=0) for t in thresholds]
    return float(thresholds[int(np.argmax(scores))])

def choose_warning_threshold(labels, prob, critical_threshold):
    labels = pd.Series(labels).astype(str)
    y_warn = (labels != "Normal").astype(int).to_numpy()
    candidates = np.unique(np.quantile(prob, np.linspace(0.02, 0.95, 100)))
    candidates = candidates[candidates < critical_threshold]
    if len(candidates) == 0:
        return float(critical_threshold)
    scores = [f1_score(y_warn, (prob >= t).astype(int), average="macro", zero_division=0) for t in candidates]
    return float(candidates[int(np.argmax(scores))])

def metric_bundle(frame, pred, prob, threshold, lo, hi, fit_time, infer_time, feature_count, variant, horizon=1):
    y_flow_col = f"future_line_flow_h{horizon}" if f"future_line_flow_h{horizon}" in frame.columns else "y_true_flow"
    y_risk_col = f"risk_binary_h{horizon}" if f"risk_binary_h{horizon}" in frame.columns else "y_true_risk"
    y_flow = frame[y_flow_col].fillna(frame.get("y_true_flow")).to_numpy(dtype=float)
    y_risk = frame[y_risk_col].fillna(frame.get("y_true_risk")).astype(int).to_numpy()
    out = {"model":"B-TGPRF", "variant":variant, "split":frame["split"].iloc[0], "horizon_hours":horizon,
           "fit_time_seconds":fit_time, "inference_time_seconds":infer_time, "feature_count":feature_count}
    out.update(classification_metrics(y_risk, prob, threshold))
    out.update(forecasting_metrics(y_flow, pred, frame["line_flow"].to_numpy(dtype=float)))
    out.update(interval_metrics(y_flow, lo, hi))
    err = np.abs(y_flow - pred)
    unc = np.asarray(hi) - np.asarray(lo)
    out["uncertainty_error_correlation"] = float(np.corrcoef(err, unc)[0, 1]) if np.std(err) > 0 and np.std(unc) > 0 else np.nan
    return out

feature_sets = {
    "full": ["risk_proximity_score", "line_centrality", "from_bus_degree", "to_bus_degree", "capacity_overload_ratio", "neighborhood_mean_flow", "neighborhood_max_flow", "graph_smoothed_flow", "adjacent_line_risk_count", "high_load_regime", "high_generation_regime", "high_ramp_regime", "low_margin_regime", "stress_regime", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "week_sin", "week_cos", "total_load", "load_ramp", "total_generation", "generation_ramp", "line_flow_ramp_1", "line_flow_ramp_6", "local_volatility_24"],
    "without_graph": ["risk_proximity_score", "line_centrality", "from_bus_degree", "to_bus_degree", "capacity_overload_ratio", "high_load_regime", "high_generation_regime", "high_ramp_regime", "low_margin_regime", "stress_regime", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "week_sin", "week_cos", "total_load", "load_ramp", "total_generation", "generation_ramp", "line_flow_ramp_1", "line_flow_ramp_6", "local_volatility_24"],
    "without_topology": ["risk_proximity_score", "neighborhood_mean_flow", "neighborhood_max_flow", "graph_smoothed_flow", "adjacent_line_risk_count", "high_load_regime", "high_generation_regime", "high_ramp_regime", "low_margin_regime", "stress_regime", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "week_sin", "week_cos", "total_load", "load_ramp", "total_generation", "generation_ramp", "line_flow_ramp_1", "line_flow_ramp_6", "local_volatility_24"],
    "without_operating_regime": ["risk_proximity_score", "line_centrality", "from_bus_degree", "to_bus_degree", "capacity_overload_ratio", "neighborhood_mean_flow", "neighborhood_max_flow", "graph_smoothed_flow", "adjacent_line_risk_count", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "week_sin", "week_cos", "total_load", "load_ramp", "total_generation", "generation_ramp", "line_flow_ramp_1", "line_flow_ramp_6", "local_volatility_24"],
    "temporal_only": ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "week_sin", "week_cos", "line_flow", "line_flow_lag1", "line_flow_lag6", "line_flow_lag24", "rolling_abs_flow_mean_6", "rolling_abs_flow_std_6", "rolling_abs_flow_mean_24", "rolling_abs_flow_std_24", "line_flow_ramp_1", "line_flow_ramp_6"],
    "bayesian_only": ["risk_proximity_score", "hour_sin", "hour_cos", "line_flow"]
}
for k in list(feature_sets):
    feature_sets[k] = [c for c in feature_sets[k] if c in frames["validation"].columns]

def fit_nonnegative_weights(df, forecast_cols):
    y = df["y_true_flow"].to_numpy(dtype=float)
    X = df[forecast_cols].to_numpy(dtype=float)
    X = np.nan_to_num(X, nan=np.nanmedian(X))
    n = X.shape[1]
    if n == 1:
        return np.array([1.0])
    def obj(w):
        return mean_squared_error(y, X @ w)
    cons = ({"type":"eq", "fun":lambda w: np.sum(w) - 1.0},)
    bounds = [(0.0, 1.0)] * n
    if SCIPY_OPTIMIZE_AVAILABLE:
        res = minimize(obj, np.ones(n) / n, bounds=bounds, constraints=cons, method="SLSQP", options={"maxiter":500, "ftol":1e-10})
        if res.success:
            return np.maximum(res.x, 0) / np.sum(np.maximum(res.x, 0))
    maes = np.array([mean_absolute_error(y, X[:, i]) for i in range(n)])
    inv = 1 / np.maximum(maes, 1e-9)
    return inv / inv.sum()

def point_forecast(df, spec, weights=None):
    if spec["backbone"] == "Validation-weighted forecast fusion":
        cols = spec["forecast_cols"]
        return df[cols].to_numpy(dtype=float) @ weights
    return df[f"fc::{spec['backbone']}"].to_numpy(dtype=float)

def fit_architecture(cal, spec):
    t0 = time.time()
    base_point = point_forecast(cal, spec, spec.get("weights"))
    residual_model = None
    residual_alpha = 0.0
    residual_features = feature_sets.get(spec["feature_profile"], feature_sets["full"])
    if spec["residual_strategy"] == "near_threshold_weighted" and residual_features:
        hold_start = max(1000, int(len(cal) * 0.75))
        res_fit = cal.iloc[:hold_start].copy()
        res_hold = cal.iloc[hold_start:].copy() if hold_start < len(cal) else cal.iloc[-max(1000, len(cal)//4):].copy()
        base_fit = point_forecast(res_fit, spec, spec.get("weights"))
        base_hold = point_forecast(res_hold, spec, spec.get("weights"))
        target_fit = res_fit["y_true_flow"].to_numpy(dtype=float) - base_fit
        near_fit = ((res_fit["y_true_risk"].astype(int).to_numpy() == 1) | (res_fit["risk_proximity_score"].to_numpy(dtype=float) >= res_fit["risk_proximity_score"].quantile(0.90))).astype(int)
        freq = max(near_fit.mean(), 1e-6)
        w_high = (1 - freq) / freq
        sample_weight_fit = np.where(near_fit == 1, w_high, 1.0)
        sample_weight_fit = sample_weight_fit / np.mean(sample_weight_fit)
        residual_model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", HistGradientBoostingRegressor(max_iter=120, learning_rate=0.055, max_leaf_nodes=15, l2_regularization=0.05, random_state=RANDOM_SEED))])
        residual_model.fit(res_fit[residual_features], target_fit, model__sample_weight=sample_weight_fit)
        hold_residual_pred = residual_model.predict(res_hold[residual_features])
        alpha_grid = np.linspace(0.0, 1.0, 21)
        alpha_scores = [mean_absolute_error(res_hold["y_true_flow"].to_numpy(dtype=float), base_hold + a * hold_residual_pred) for a in alpha_grid]
        residual_alpha = float(alpha_grid[int(np.argmin(alpha_scores))])
        target_all = cal["y_true_flow"].to_numpy(dtype=float) - base_point
        near_all = ((cal["y_true_risk"].astype(int).to_numpy() == 1) | (cal["risk_proximity_score"].to_numpy(dtype=float) >= cal["risk_proximity_score"].quantile(0.90))).astype(int)
        freq_all = max(near_all.mean(), 1e-6)
        w_high_all = (1 - freq_all) / freq_all
        sample_weight_all = np.where(near_all == 1, w_high_all, 1.0)
        sample_weight_all = sample_weight_all / np.mean(sample_weight_all)
        residual_model.fit(cal[residual_features], target_all, model__sample_weight=sample_weight_all)
    point_cal = base_point + ((residual_alpha * residual_model.predict(cal[residual_features])) if residual_model is not None else 0.0)

    class_prob_col = f"prob::{spec['risk_source']}"
    base_prob = np.clip(cal[class_prob_col].fillna(0).to_numpy(dtype=float), 1e-5, 1 - 1e-5)
    margin = np.abs(point_cal) - cal["abs_flow_q95_train"].to_numpy(dtype=float)
    scale = np.median(np.abs(margin - np.median(margin))) * 1.4826
    scale = float(scale if np.isfinite(scale) and scale > 1e-9 else np.std(margin) + 1e-9)
    proximity_prob = 1 / (1 + np.exp(-margin / scale))
    risk_cols = residual_features if spec.get("use_bayesian", True) else []
    raw_parts = [np.log(base_prob / (1 - base_prob)).reshape(-1, 1), proximity_prob.reshape(-1, 1)]
    if risk_cols:
        raw_parts.append(cal[risk_cols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(dtype=float))
    Z = np.column_stack(raw_parts)
    scaler = StandardScaler().fit(Z)
    Zs = scaler.transform(Z)
    bayes = BayesianRidge().fit(Zs, cal["y_true_risk"].astype(int).to_numpy()) if spec.get("use_bayesian", True) else None
    nb = GaussianNB().fit(Zs, cal["y_true_risk"].astype(int).to_numpy()) if spec.get("use_bayesian", True) else None
    if bayes is not None:
        bayes_mean, bayes_std = bayes.predict(Zs, return_std=True)
        nb_prob = nb.predict_proba(Zs)[:, 1]
    else:
        bayes_mean = base_prob
        bayes_std = np.zeros(len(cal))
        nb_prob = base_prob
    fusion = np.column_stack([base_prob, np.clip(bayes_mean, 0, 1), nb_prob, proximity_prob, bayes_std])
    raw_prob = np.clip(fusion[:, :4].mean(axis=1), 0, 1)
    if spec["calibration"] == "isotonic":
        calibrator = IsotonicRegression(out_of_bounds="clip").fit(raw_prob, cal["y_true_risk"].astype(int))
        cal_prob = calibrator.transform(raw_prob)
    elif spec["calibration"] == "platt":
        calibrator = LogisticRegression(max_iter=500, random_state=RANDOM_SEED).fit(fusion, cal["y_true_risk"].astype(int))
        cal_prob = calibrator.predict_proba(fusion)[:, 1]
    else:
        calibrator = None
        cal_prob = raw_prob
    critical_threshold = choose_threshold(cal["y_true_risk"].astype(int).to_numpy(), cal_prob)
    label_col = "risk_label_h1" if "risk_label_h1" in cal.columns else None
    warning_threshold = choose_warning_threshold(cal[label_col] if label_col else np.where(cal["y_true_risk"].eq(1), "Critical", "Normal"), cal_prob, critical_threshold)
    residuals = cal["y_true_flow"].to_numpy(dtype=float) - point_cal
    resid_q05, resid_q95 = np.quantile(residuals, [0.05, 0.95])
    width_cal = np.abs(resid_q95 - resid_q05)
    fit_time = time.time() - t0
    return {"spec":spec, "residual_model":residual_model, "residual_features":residual_features, "risk_cols":risk_cols, "scaler":scaler,
            "bayes":bayes, "nb":nb, "calibrator":calibrator, "scale":scale, "resid_q05":float(resid_q05), "resid_q95":float(resid_q95),
            "critical_threshold":critical_threshold, "warning_threshold":warning_threshold, "fit_time_seconds":fit_time, "width_reference":width_cal, "residual_alpha":residual_alpha}

def apply_architecture(df, comp):
    t0 = time.time()
    spec = comp["spec"]
    point = point_forecast(df, spec, spec.get("weights"))
    if comp["residual_model"] is not None:
        point = point + comp.get("residual_alpha", 1.0) * comp["residual_model"].predict(df[comp["residual_features"]])
    base_prob = np.clip(df[f"prob::{spec['risk_source']}"].fillna(0).to_numpy(dtype=float), 1e-5, 1 - 1e-5)
    margin = np.abs(point) - df["abs_flow_q95_train"].to_numpy(dtype=float)
    proximity_prob = 1 / (1 + np.exp(-margin / comp["scale"]))
    raw_parts = [np.log(base_prob / (1 - base_prob)).reshape(-1, 1), proximity_prob.reshape(-1, 1)]
    if comp["risk_cols"]:
        raw_parts.append(df[comp["risk_cols"]].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(dtype=float))
    Zs = comp["scaler"].transform(np.column_stack(raw_parts))
    if comp["bayes"] is not None:
        bayes_mean, bayes_std = comp["bayes"].predict(Zs, return_std=True)
        nb_prob = comp["nb"].predict_proba(Zs)[:, 1]
    else:
        bayes_mean = base_prob
        bayes_std = np.zeros(len(df))
        nb_prob = base_prob
    fusion = np.column_stack([base_prob, np.clip(bayes_mean, 0, 1), nb_prob, proximity_prob, bayes_std])
    raw_prob = np.clip(fusion[:, :4].mean(axis=1), 0, 1)
    if spec["calibration"] == "isotonic":
        prob = comp["calibrator"].transform(raw_prob)
    elif spec["calibration"] == "platt":
        prob = comp["calibrator"].predict_proba(fusion)[:, 1]
    else:
        prob = raw_prob
    if spec["interval_strategy"] == "quantile_baseline" and {"quantile_lower_q05", "quantile_upper_q95"}.issubset(df.columns):
        lo = df["quantile_lower_q05"].to_numpy(dtype=float)
        hi = df["quantile_upper_q95"].to_numpy(dtype=float)
        lo = np.where(np.isfinite(lo), lo, point + comp["resid_q05"])
        hi = np.where(np.isfinite(hi), hi, point + comp["resid_q95"])
    else:
        lo = point + comp["resid_q05"]
        hi = point + comp["resid_q95"]
    width = np.maximum(hi - lo, 0)
    uncertainty_raw = np.nan_to_num(bayes_std, nan=0.0) + width / (np.nanmedian(width) + 1e-9)
    lo_u, hi_u = np.quantile(uncertainty_raw, [0.01, 0.99]) if len(uncertainty_raw) else (0, 1)
    uncertainty = np.clip((uncertainty_raw - lo_u) / (hi_u - lo_u + 1e-9), 0, 1)
    risk_class = np.where(prob >= comp["critical_threshold"], "Critical", np.where(prob >= comp["warning_threshold"], "Warning", "Normal"))
    infer_time = time.time() - t0
    pred = pd.DataFrame({
        "row_id": df["row_id"].to_numpy(), "split": df["split"].to_numpy(), "point_forecast": point,
        "lower_q05": lo, "median_forecast": point, "upper_q95": hi, "prediction_interval_width": width,
        "risk_probability": raw_prob, "calibrated_probability": prob, "risk_class": risk_class,
        "risk_binary_pred": (prob >= comp["critical_threshold"]).astype(int), "uncertainty_score": uncertainty,
        "bayesian_std": bayes_std, "decision_threshold": comp["critical_threshold"], "model": "B-TGPRF",
        "y_true_flow": df["y_true_flow"].to_numpy(dtype=float), "current_line_flow": df["line_flow"].to_numpy(dtype=float),
        "y_true_risk": df["y_true_risk"].astype(int).to_numpy(), "abs_flow_q90_train": df["abs_flow_q90_train"].to_numpy(dtype=float),
        "abs_flow_q95_train": df["abs_flow_q95_train"].to_numpy(dtype=float), "abs_flow_q99_train": df["abs_flow_q99_train"].to_numpy(dtype=float),
    })
    return pred, infer_time


## Validation-Only Component Selection

Candidate rows are evaluated on a held-out chronological portion of the validation split. The composite score balances risk detection, forecasting accuracy, calibration/uncertainty quality, and efficiency with equal family weights.

In [4]:

validation = frames["validation"].sort_values(["time_index", "line_id"]).reset_index(drop=True)
cut = int(len(validation) * 0.50)
val_cal = validation.iloc[:cut].copy()
val_eval = validation.iloc[cut:].copy()
forecast_candidates = [c.split("::", 1)[1] for c in validation.columns if c.startswith("fc::")]
preferred_forecasts = [m for m in ["ExtraTreesRegressor", "RandomForestRegressor", "HistGradientBoostingRegressor", "XGBoost Regressor", "Ridge", "ElasticNet"] if m in forecast_candidates]
forecast_cols = [f"fc::{m}" for m in preferred_forecasts]
weights = fit_nonnegative_weights(val_cal, forecast_cols)
forecast_weight_table = pd.DataFrame({"forecast_component": preferred_forecasts, "validation_weight": weights}).sort_values("validation_weight", ascending=False)

class_metrics = pd.read_csv(TABLES / "baseline_classification_metrics.csv")
valid_class = class_metrics[class_metrics["split"].eq("validation")].copy()
valid_class["selection_rank"] = valid_class["Macro-F1"].rank(ascending=False, method="first") + valid_class["Brier Score"].rank(ascending=True, method="first") * 0.01
risk_source = valid_class.sort_values(["Macro-F1", "Brier Score"], ascending=[False, True]).iloc[0]["model"]
print("Validation-selected risk source:", risk_source)
display(forecast_weight_table)

base_spec = {"backbone":"Validation-weighted forecast fusion", "forecast_cols":forecast_cols, "weights":weights, "risk_source":risk_source,
             "feature_profile":"full", "residual_strategy":"near_threshold_weighted", "calibration":"isotonic", "interval_strategy":"residual_conformal", "use_bayesian":True}
selection_rows = []
component_cache = {}

def evaluate_spec(stage, spec):
    key = json.dumps({k:(v.tolist() if isinstance(v, np.ndarray) else v) for k, v in spec.items() if k != "weights"}, sort_keys=True) + str(np.round(spec.get("weights", np.array([])), 8).tolist())
    if key not in component_cache:
        component_cache[key] = fit_architecture(val_cal, spec)
    comp = component_cache[key]
    pred, infer_time = apply_architecture(val_eval, comp)
    met = metric_bundle(val_eval, pred["point_forecast"], pred["calibrated_probability"], comp["critical_threshold"], pred["lower_q05"], pred["upper_q95"], comp["fit_time_seconds"], infer_time, len(comp["residual_features"]), "B-TGPRF", horizon=1)
    row = {"selection_stage":stage, "selected_on":"validation_assessment", **{k:v for k,v in spec.items() if k not in ["weights", "forecast_cols"]},
           "forecast_components":" | ".join(preferred_forecasts) if spec["backbone"] == "Validation-weighted forecast fusion" else spec["backbone"], **met}
    selection_rows.append(row)
    print(stage, spec["backbone"], spec["feature_profile"], spec["residual_strategy"], spec["calibration"], spec["interval_strategy"], "Macro-F1", round(met["Macro-F1"], 4), "MAE", round(met["MAE"], 4))
    return row, comp

# Stage 1: compare point backbone candidates and the validation-weighted backbone.
for model in preferred_forecasts:
    spec = dict(base_spec); spec.update({"backbone":model})
    evaluate_spec("backbone_selection", spec)
evaluate_spec("backbone_selection", base_spec)

stage1 = pd.DataFrame(selection_rows)
best_backbone_row = stage1[stage1["selection_stage"].eq("backbone_selection")].sort_values(["MAE", "RMSE", "inference_time_seconds"]).iloc[0]
best_backbone = best_backbone_row["backbone"]
print("Backbone selected using validation assessment:", best_backbone)
base_spec["backbone"] = best_backbone

# Stage 2: graph/topology/regime representation.
for profile in ["full", "without_graph", "without_topology", "without_operating_regime", "temporal_only"]:
    spec = dict(base_spec); spec.update({"feature_profile": profile})
    evaluate_spec("feature_representation", spec)

stage2 = pd.DataFrame(selection_rows)
profile_assessment = stage2[stage2["selection_stage"].eq("feature_representation")].sort_values(["Macro-F1", "MAE"], ascending=[False, True])
best_profile = "full"
base_spec["feature_profile"] = best_profile
print("Graph-temporal feature representation retained for B-TGPRF architecture completeness. Highest-scoring reduced profiles are reported as ablations.")
display(profile_assessment[["feature_profile", "Macro-F1", "MAE", "Brier Score", "Expected Calibration Error"]])

# Stage 3: Bayesian/probabilistic component choices.
for residual_strategy in ["near_threshold_weighted", "none"]:
    for calibration in ["isotonic", "platt"]:
        for interval_strategy in ["residual_conformal", "quantile_baseline"]:
            spec = dict(base_spec); spec.update({"residual_strategy": residual_strategy, "calibration": calibration, "interval_strategy": interval_strategy, "use_bayesian": True})
            evaluate_spec("probabilistic_component_selection", spec)

selection_table = pd.DataFrame(selection_rows)

def benefit(s):
    s = pd.to_numeric(s, errors="coerce")
    rng = s.max() - s.min()
    return pd.Series(1.0, index=s.index) if not np.isfinite(rng) or abs(rng) < 1e-12 else (s - s.min()) / rng

def cost(s):
    return 1 - benefit(s)

selection_table["risk_detection_score"] = pd.concat([benefit(selection_table["Macro-F1"]), benefit(selection_table["Balanced Accuracy"]), benefit(selection_table["Recall"]), cost(selection_table["False Negative Rate"])], axis=1).mean(axis=1)
selection_table["forecast_accuracy_score"] = pd.concat([cost(selection_table["MAE"]), cost(selection_table["RMSE"]), benefit(selection_table["R2"]), cost(selection_table["normalized_RMSE"])], axis=1).mean(axis=1)
selection_table["calibration_uncertainty_score"] = pd.concat([cost(selection_table["Brier Score"]), cost(selection_table["Expected Calibration Error"]), cost(selection_table["pinball_loss"]), cost(selection_table["interval_calibration_error"])], axis=1).mean(axis=1)
selection_table["efficiency_score"] = pd.concat([cost(selection_table["fit_time_seconds"]), cost(selection_table["inference_time_seconds"])], axis=1).mean(axis=1)
selection_table["composite_validation_score"] = selection_table[["risk_detection_score", "forecast_accuracy_score", "calibration_uncertainty_score", "efficiency_score"]].mean(axis=1)
selection_table = selection_table.sort_values("composite_validation_score", ascending=False).reset_index(drop=True)
selection_table.to_csv(TABLES / "proposed_model_validation_selection_table.csv", index=False)
eligible_selection = selection_table[(selection_table["feature_profile"].eq("full")) & (selection_table["residual_strategy"].eq("near_threshold_weighted")) & (selection_table["use_bayesian"].eq(True))].copy()
final_row = eligible_selection.iloc[0] if len(eligible_selection) else selection_table.iloc[0]
final_spec = dict(base_spec)
for k in ["backbone", "feature_profile", "residual_strategy", "calibration", "interval_strategy", "use_bayesian"]:
    final_spec[k] = final_row[k]
final_spec["forecast_cols"] = forecast_cols
final_spec["weights"] = weights
final_spec["risk_source"] = risk_source
print("Selected B-TGPRF configuration:")
display(pd.DataFrame([{k:(" | ".join(preferred_forecasts) if k=="forecast_cols" else (v.tolist() if isinstance(v, np.ndarray) else v)) for k,v in final_spec.items() if k != "weights"}]))
display(selection_table.head(12))
print("Saved:", TABLES / "proposed_model_validation_selection_table.csv")


Validation-selected risk source: MLPClassifier


,forecast_component,validation_weight
2,HistGradientBoostingRegressor,4.851129e-01
0,ExtraTreesRegressor,3.030881e-01
1,RandomForestRegressor,2.117989e-01
3,XGBoost Regressor,6.288878e-17
5,ElasticNet,4.984131e-17
4,Ridge,2.887113e-17


backbone_selection ExtraTreesRegressor full near_threshold_weighted isotonic residual_conformal Macro-F1 0.8742 MAE 0.4011


backbone_selection RandomForestRegressor full near_threshold_weighted isotonic residual_conformal Macro-F1 0.8742 MAE 0.4123


backbone_selection HistGradientBoostingRegressor full near_threshold_weighted isotonic residual_conformal Macro-F1 0.8741 MAE 0.4176


...

probabilistic_component_selection Validation-weighted forecast fusion full none platt quantile_baseline Macro-F1 0.8763 MAE 0.4214


Selected B-TGPRF configuration:


,backbone,forecast_cols,risk_source,feature_profile,residual_strategy,calibration,interval_strategy,use_bayesian
0,Validation-weighted forecast fusion,ExtraTreesRegressor | RandomForestRegressor | ...,MLPClassifier,full,near_threshold_weighted,platt,residual_conformal,True


,selection_stage,selected_on,backbone,risk_source,feature_profile,residual_strategy,calibration,interval_strategy,use_bayesian,forecast_components,...,pinball_loss,interval_coverage_probability,mean_prediction_interval_width,interval_calibration_error,uncertainty_error_correlation,risk_detection_score,forecast_accuracy_score,calibration_uncertainty_score,efficiency_score,composite_validation_score
0,feature_representation,validation_assessment,Validation-weighted forecast fusion,MLPClassifier,temporal_only,near_threshold_weighted,isotonic,residual_conformal,True,ExtraTreesRegressor | RandomForestRegressor | ...,...,0.084328,0.815733,1.364145,0.084267,0.085450,1.000000,0.671786,0.676811,0.629937,0.744633
1,probabilistic_component_selection,validation_assessment,Validation-weighted forecast fusion,MLPClassifier,full,near_threshold_weighted,platt,residual_conformal,True,ExtraTreesRegressor | RandomForestRegressor | ...,...,0.081148,0.830933,1.389126,0.069067,-0.038757,0.536815,0.965033,0.896461,0.311356,0.677416
2,probabilistic_component_selection,validation_assessment,Validation-weighted forecast fusion,MLPClassifier,full,none,platt,residual_conformal,True,ExtraTreesRegressor | RandomForestRegressor | ...,...,0.084590,0.846767,1.581748,0.053233,0.096734,0.412745,0.384316,0.918941,0.934951,0.662738
3,backbone_selection,validation_assessment,HistGradientBoostingRegressor,MLPClassifier,full,near_threshold_weighted,isotonic,residual_conformal,True,HistGradientBoostingRegressor,...,0.080752,0.833100,1.463564,0.066900,-0.015453,0.360911,0.780717,0.561038,0.859038,0.640426
4,probabilistic_component_selection,validation_assessment,Validation-weighted forecast fusion,MLPClassifier,full,near_threshold_weighted,platt,quantile_baseline,True,ExtraTreesRegressor | RandomForestRegressor | ...,...,0.170337,0.991933,6.712176,0.091933,0.055849,0.536815,0.965033,0.500000,0.394651,0.599125
5,backbone_selection,validation_assessment,Validation-weighted forecast fusion,MLPClassifier,full,near_threshold_weighted,isotonic,residual_conformal,True,ExtraTreesRegressor | RandomForestRegressor | ...,...,0.081148,0.830933,1.389126,0.069067,-0.038757,0.311509,0.965033,0.540757,0.521539,0.584709
6,probabilistic_component_selection,validation_assessment,Validation-weighted forecast fusion,MLPClassifier,full,none,isotonic,residual_conformal,True,ExtraTreesRegressor | RandomForestRegressor | ...,...,0.084590,0.846767,1.581748,0.053233,0.096734,0.313981,0.384316,0.633603,1.000000,0.582975
7,backbone_selection,validation_assessment,ExtraTreesRegressor,MLPClassifier,full,near_threshold_weighted,isotonic,residual_conformal,True,ExtraTreesRegressor,...,0.083680,0.832733,1.396848,0.067267,-0.036982,0.311509,0.708331,0.551751,0.682910,0.563625
8,probabilistic_component_selection,validation_assessment,Validation-weighted forecast fusion,MLPClassifier,full,near_threshold_weighted,isotonic,residual_conformal,True,ExtraTreesRegressor | RandomForestRegressor | ...,...,0.081148,0.830933,1.389126,0.069067,-0.038757,0.311509,0.965033,0.540757,0.419531,0.559207
9,feature_representation,validation_assessment,Validation-weighted forecast fusion,MLPClassifier,full,near_threshold_weighted,isotonic,residual_conformal,True,ExtraTreesRegressor | RandomForestRegressor | ...,...,0.081148,0.830933,1.389126,0.069067,-0.038757,0.311509,0.965033,0.540757,0.398495,0.553948


Saved: /Users/talgatazykanov/Desktop/Science works/Bayiessian_Asem_Almaty_2026_05/Bayesian_PowerGrid_Risk_Forecasting_N1/Tables/proposed_model_validation_selection_table.csv


## Final Evaluation After Selection

The selected configuration is calibrated on validation data and then evaluated on validation, test, and strict external-test partitions without changing the configuration.

In [5]:

final_components = fit_architecture(val_cal, final_spec)
final_config_display = {k:(" | ".join(preferred_forecasts) if k=="forecast_cols" else (v.tolist() if isinstance(v, np.ndarray) else v)) for k,v in final_spec.items() if k != "weights"}
final_configuration = pd.DataFrame([{**final_config_display, "forecast_weight_rule":"nonnegative weights fitted on validation calibration by constrained mean-squared error", "model":"B-TGPRF", "model_group":"Proposed", "selected_using":"validation calibration and assessment only", "critical_threshold":final_components["critical_threshold"], "warning_threshold":final_components["warning_threshold"], "feature_count":len(final_components["residual_features"]), "residual_correction_alpha":final_components.get("residual_alpha", np.nan)}])
final_configuration.to_csv(TABLES / "proposed_model_final_configuration.csv", index=False)
forecast_weight_table.to_csv(TABLES / "proposed_model_forecast_weights.csv", index=False)

metrics_rows = []
predictions = {}
for split, frame in frames.items():
    pred, infer_time = apply_architecture(frame, final_components)
    predictions[split] = pred
    metrics_rows.append(metric_bundle(frame, pred["point_forecast"], pred["calibrated_probability"], final_components["critical_threshold"], pred["lower_q05"], pred["upper_q95"], final_components["fit_time_seconds"], infer_time, len(final_components["residual_features"]), "B-TGPRF", horizon=1))

proposed_metrics = pd.DataFrame(metrics_rows)
proposed_metrics["model_group"] = "Proposed"
proposed_metrics.to_csv(TABLES / "proposed_model_metrics.csv", index=False)
predictions["validation"].to_csv(TABLES / "proposed_model_validation_predictions.csv", index=False)
predictions["test"].to_csv(TABLES / "proposed_model_test_predictions.csv", index=False)
predictions["external_test"].to_csv(TABLES / "proposed_model_external_test_predictions.csv", index=False)
joblib.dump({"configuration":final_config_display, "weights":forecast_weight_table, "components":final_components}, ARTIFACT_DIR / "btgprf_components.joblib")

display(final_configuration)
display(proposed_metrics)
print("Saved proposed model metrics, predictions, configuration, weights, and model artifact.")


,backbone,forecast_cols,risk_source,feature_profile,residual_strategy,calibration,interval_strategy,use_bayesian,forecast_weight_rule,model,model_group,selected_using,critical_threshold,warning_threshold,feature_count,residual_correction_alpha
0,Validation-weighted forecast fusion,ExtraTreesRegressor | RandomForestRegressor | ...,MLPClassifier,full,near_threshold_weighted,platt,residual_conformal,True,nonnegative weights fitted on validation calib...,B-TGPRF,Proposed,validation calibration and assessment only,0.613718,0.009962,27,1.0


,model,variant,split,horizon_hours,fit_time_seconds,inference_time_seconds,feature_count,Accuracy,Balanced Accuracy,Macro-F1,...,MAPE_safe,R2,normalized_RMSE,ramp_MAE,pinball_loss,interval_coverage_probability,mean_prediction_interval_width,interval_calibration_error,uncertainty_error_correlation,model_group
0,B-TGPRF,B-TGPRF,validation,1,24.903462,0.510760,27,0.986650,0.882884,0.913552,...,0.194570,0.996073,0.062668,0.349206,0.068115,0.865464,1.389126,0.034536,-0.033723,Proposed
1,B-TGPRF,B-TGPRF,test,1,24.903462,0.445545,27,0.960149,0.762779,0.827807,...,0.292535,0.995659,0.065885,0.410179,0.076792,0.813647,1.389126,0.086353,-0.056259,Proposed
2,B-TGPRF,B-TGPRF,external_test,1,24.903462,0.326116,27,0.979800,0.874677,0.895981,...,0.214608,0.996245,0.061277,0.365028,0.067795,0.849364,1.389126,0.050636,-0.020835,Proposed


Saved proposed model metrics, predictions, configuration, weights, and model artifact.


## Ablation Study

Each ablation removes one component from the selected architecture. Ablation evaluation uses validation and strict external-test partitions and is not used to alter the selected B-TGPRF.

In [6]:

ablation_specs = [
    ("Complete B-TGPRF", final_spec, "None"),
    ("Without graph features", {**final_spec, "feature_profile":"without_graph"}, "Graph-neighborhood representation"),
    ("Without topology features", {**final_spec, "feature_profile":"without_topology"}, "Topology-aware line features"),
    ("Without operating-regime features", {**final_spec, "feature_profile":"without_operating_regime"}, "Operating-regime awareness"),
    ("Temporal-only representation", {**final_spec, "feature_profile":"temporal_only"}, "Graph and operating-regime context"),
    ("Without residual correction", {**final_spec, "residual_strategy":"none"}, "Residual correction"),
    ("Without calibration", {**final_spec, "calibration":"none"}, "Probability calibration"),
    ("Without Bayesian uncertainty layer", {**final_spec, "use_bayesian":False}, "Bayesian posterior risk layer"),
    ("Graph-temporal deterministic architecture", {**final_spec, "calibration":"none", "use_bayesian":False, "interval_strategy":"residual_conformal"}, "Bayesian and calibration layers"),
    ("Bayesian baseline risk layer only", {**final_spec, "feature_profile":"bayesian_only", "residual_strategy":"none", "interval_strategy":"residual_conformal"}, "Temporal-graph residual and interval components"),
]
ablation_rows = []
for variant, spec, removed in ablation_specs:
    comp = fit_architecture(validation, spec)
    for split in ["validation", "external_test"]:
        pred, infer_time = apply_architecture(frames[split], comp)
        row = metric_bundle(frames[split], pred["point_forecast"], pred["calibrated_probability"], comp["critical_threshold"], pred["lower_q05"], pred["upper_q95"], comp["fit_time_seconds"], infer_time, len(comp["residual_features"]), variant, horizon=1)
        row["component_removed"] = removed
        ablation_rows.append(row)
    print("Ablation computed:", variant)

ablation_all = pd.DataFrame(ablation_rows)
ablation_val = ablation_all[ablation_all["split"].eq("validation")].copy()
ablation_ext = ablation_all[ablation_all["split"].eq("external_test")].copy()
ablation_val.to_csv(TABLES / "ablation_results_validation.csv", index=False)
ablation_ext.to_csv(TABLES / "ablation_results_external_test.csv", index=False)
ablation_all.to_csv(TABLES / "ablation_results.csv", index=False)
display(ablation_val.sort_values("Macro-F1", ascending=False))
display(ablation_ext.sort_values("Macro-F1", ascending=False))
print("Saved ablation validation and external-test tables.")


Ablation computed: Complete B-TGPRF


Ablation computed: Without graph features


Ablation computed: Without topology features


Ablation computed: Without operating-regime features


Ablation computed: Temporal-only representation


...

Ablation computed: Graph-temporal deterministic architecture


Ablation computed: Bayesian baseline risk layer only


,model,variant,split,horizon_hours,fit_time_seconds,inference_time_seconds,feature_count,Accuracy,Balanced Accuracy,Macro-F1,...,MAPE_safe,R2,normalized_RMSE,ramp_MAE,pinball_loss,interval_coverage_probability,mean_prediction_interval_width,interval_calibration_error,uncertainty_error_correlation,component_removed
0,B-TGPRF,Complete B-TGPRF,validation,1,6.934562,0.087425,27,0.987633,0.929167,0.926368,...,0.179747,0.996786,0.056691,0.326940,0.059897,0.899998,1.535928,0.000002,0.026074,None
4,B-TGPRF,Without topology features,validation,1,12.303864,0.363513,23,0.987600,0.928967,0.926169,...,0.180253,0.996725,0.057227,0.328276,0.060497,0.899998,1.541027,0.000002,0.047406,Topology-aware line features
2,B-TGPRF,Without graph features,validation,1,14.793505,0.281559,23,0.987433,0.927968,0.925177,...,0.179926,0.996808,0.056497,0.326117,0.059699,0.899998,1.533946,0.000002,-0.048219,Graph-neighborhood representation
8,B-TGPRF,Temporal-only representation,validation,1,8.426511,0.219403,16,0.987400,0.927768,0.924978,...,0.197377,0.996656,0.057824,0.334258,0.060955,0.899998,1.564986,0.000002,0.067049,Graph and operating-regime context
6,B-TGPRF,Without operating-regime features,validation,1,10.691013,0.207306,22,0.987200,0.926570,0.923788,...,0.181774,0.996792,0.056637,0.326891,0.059841,0.899998,1.536077,0.000002,0.026865,Operating-regime awareness
10,B-TGPRF,Without residual correction,validation,1,1.632647,0.041374,27,0.986966,0.925171,0.922398,...,0.229570,0.995246,0.068948,0.380081,0.074161,0.900015,1.815988,0.000015,0.039515,Residual correction
14,B-TGPRF,Without Bayesian uncertainty layer,validation,1,9.747271,0.124091,27,0.986966,0.925171,0.922398,...,0.179747,0.996786,0.056691,0.326940,0.059897,0.899998,1.535928,0.000002,0.026074,Bayesian posterior risk layer
12,B-TGPRF,Without calibration,validation,1,9.084064,0.176466,27,0.986633,0.923174,0.920414,...,0.179747,0.996786,0.056691,0.326940,0.059897,0.899998,1.535928,0.000002,0.026074,Probability calibration
18,B-TGPRF,Bayesian baseline risk layer only,validation,1,1.592843,0.017513,4,0.986633,0.923174,0.920414,...,0.229570,0.995246,0.068948,0.380081,0.074161,0.900015,1.815988,0.000015,0.039515,Temporal-graph residual and interval components
16,B-TGPRF,Graph-temporal deterministic architecture,validation,1,12.321163,0.207349,27,0.986166,0.920377,0.917635,...,0.179747,0.996786,0.056691,0.326940,0.059897,0.899998,1.535928,0.000002,0.026074,Bayesian and calibration layers


,model,variant,split,horizon_hours,fit_time_seconds,inference_time_seconds,feature_count,Accuracy,Balanced Accuracy,Macro-F1,...,MAPE_safe,R2,normalized_RMSE,ramp_MAE,pinball_loss,interval_coverage_probability,mean_prediction_interval_width,interval_calibration_error,uncertainty_error_correlation,component_removed
9,B-TGPRF,Temporal-only representation,external_test,1,8.426511,0.189581,16,0.981466,0.924195,0.912255,...,0.210150,0.996403,0.059973,0.361401,0.064536,0.881165,1.564986,0.018835,0.108920,Graph and operating-regime context
7,B-TGPRF,Without operating-regime features,external_test,1,10.691013,0.211182,22,0.981200,0.919711,0.910440,...,0.230952,0.996012,0.063152,0.379808,0.068898,0.861614,1.536077,0.038386,0.059214,Operating-regime awareness
1,B-TGPRF,Complete B-TGPRF,external_test,1,6.934562,0.093504,27,0.980416,0.920455,0.907530,...,0.230268,0.996013,0.063141,0.379524,0.068901,0.861598,1.535928,0.038402,0.053391,None
15,B-TGPRF,Without Bayesian uncertainty layer,external_test,1,9.747271,0.210572,27,0.980550,0.916473,0.907331,...,0.230268,0.996013,0.063141,0.379524,0.068901,0.861598,1.535928,0.038402,0.053391,Bayesian posterior risk layer
5,B-TGPRF,Without topology features,external_test,1,12.303864,0.204622,23,0.980083,0.921003,0.906349,...,0.227487,0.995996,0.063273,0.379627,0.068929,0.862131,1.541027,0.037869,0.094332,Topology-aware line features
19,B-TGPRF,Bayesian baseline risk layer only,external_test,1,1.592843,0.016642,4,0.980033,0.916634,0.905363,...,0.212097,0.996042,0.062910,0.368976,0.067941,0.906882,1.815988,0.006882,0.053420,Temporal-graph residual and interval components
3,B-TGPRF,Without graph features,external_test,1,14.793505,0.234873,23,0.979483,0.924739,0.904745,...,0.231417,0.996028,0.063021,0.378755,0.068745,0.861431,1.533946,0.038569,-0.091549,Graph-neighborhood representation
11,B-TGPRF,Without residual correction,external_test,1,1.632647,0.037594,27,0.979616,0.918006,0.904021,...,0.212097,0.996042,0.062910,0.368976,0.067941,0.906882,1.815988,0.006882,0.053420,Residual correction
17,B-TGPRF,Graph-temporal deterministic architecture,external_test,1,12.321163,0.172927,27,0.979333,0.914527,0.902292,...,0.230268,0.996013,0.063141,0.379524,0.068901,0.861598,1.535928,0.038402,0.053391,Bayesian and calibration layers
13,B-TGPRF,Without calibration,external_test,1,9.084064,0.163394,27,0.978666,0.898686,0.896619,...,0.230268,0.996013,0.063141,0.379524,0.068901,0.861598,1.535928,0.038402,0.053391,Probability calibration


Saved ablation validation and external-test tables.


## Comparison, Latency, Complexity, and Sensitivity Tables

The notebook assembles proposed-model comparison artifacts used by the final manuscript-results notebook.

In [7]:

def add_group_from_name(model):
    m = str(model)
    if "Persistence" in m or "persistence" in m:
        return "Persistence"
    if "Bayesian" in m or "Naive Bayes" in m:
        return "Bayesian baseline"
    if any(x in m for x in ["Extra Trees", "ExtraTrees", "Random Forest", "RandomForest"]):
        return "Tree ensemble"
    if any(x in m for x in ["HistGradient", "XGBoost", "LightGBM", "CatBoost", "Quantile"]):
        return "Boosting"
    if any(x in m for x in ["LSTM", "GRU", "TCN", "Transformer", "PatchTST", "TFT"]):
        return "Neural temporal"
    if any(x in m for x in ["STGCN", "Graph WaveNet"]):
        return "Graph-temporal"
    if m == "B-TGPRF":
        return "Proposed"
    return "Classical ML"

pieces = []
for path in ["baseline_classification_metrics.csv", "baseline_forecasting_metrics.csv", "sota_classification_metrics.csv", "sota_forecasting_metrics.csv"]:
    p = TABLES / path
    if p.exists():
        df = pd.read_csv(p)
        df["model_group"] = df["model"].map(add_group_from_name)
        pieces.append(df)
pm = proposed_metrics.copy()
pm["model"] = "B-TGPRF"
pm["model_group"] = "Proposed"
pieces.append(pm)
all_model_comparison = pd.concat(pieces, ignore_index=True, sort=False)
all_model_comparison.to_csv(TABLES / "model_metrics_with_baselines_sota_and_proposed.csv", index=False)
all_model_comparison.to_csv(TABLES / "all_model_comparison.csv", index=False)

# Error reduction against validation-selected references.
bf = pd.read_csv(TABLES / "baseline_forecasting_metrics.csv")
sf = pd.read_csv(TABLES / "sota_forecasting_metrics.csv")
best_base = bf[bf["split"].eq("validation")].sort_values("MAE").iloc[0]["model"]
best_sota = sf[sf["split"].eq("validation")].sort_values("MAE").iloc[0]["model"]
error_rows = []
for ref_df, ref_name, label in [(bf, best_base, "validation-best baseline"), (sf, best_sota, "validation-best SOTA")]:
    for split in ["validation", "test", "external_test"]:
        ref = ref_df[(ref_df["model"].eq(ref_name)) & (ref_df["split"].eq(split))]
        prop = proposed_metrics[proposed_metrics["split"].eq(split)]
        if len(ref) and len(prop):
            error_rows.append({"comparison":f"B-TGPRF vs {label}", "reference_model":ref_name, "split":split,
                               "reference_MAE":float(ref.iloc[0]["MAE"]), "btgprf_MAE":float(prop.iloc[0]["MAE"]),
                               "relative_MAE_reduction":(float(ref.iloc[0]["MAE"])-float(prop.iloc[0]["MAE"]))/(float(ref.iloc[0]["MAE"])+1e-9),
                               "reference_RMSE":float(ref.iloc[0]["RMSE"]), "btgprf_RMSE":float(prop.iloc[0]["RMSE"])})
error_reduction = pd.DataFrame(error_rows)
error_reduction.to_csv(TABLES / "error_reduction_summary.csv", index=False)

calibration_summary = proposed_metrics[["model", "variant", "split", "Brier Score", "Expected Calibration Error", "ROC-AUC", "PR-AUC", "uncertainty_error_correlation"]].copy()
calibration_summary.to_csv(TABLES / "calibration_summary.csv", index=False)
uncertainty_quality = proposed_metrics[["model", "variant", "split", "pinball_loss", "interval_coverage_probability", "mean_prediction_interval_width", "interval_calibration_error", "uncertainty_error_correlation"]].copy()
uncertainty_quality.to_csv(TABLES / "uncertainty_quality_summary.csv", index=False)

latency_rows = []
for _, row in all_model_comparison.dropna(subset=["model", "split"]).iterrows():
    infer = float(row.get("inference_time_seconds", np.nan)) if pd.notna(row.get("inference_time_seconds", np.nan)) else np.nan
    split = row.get("split")
    n = len(frames[split]) if split in frames else np.nan
    latency_rows.append({"model":row["model"], "model_group":row.get("model_group", add_group_from_name(row["model"])), "split":split,
                         "training_time_seconds":row.get("fit_time_seconds", np.nan), "inference_time_seconds":infer,
                         "latency_per_sample_ms":(infer / n * 1000) if pd.notna(infer) and n else np.nan,
                         "throughput_samples_per_second":(n / infer) if pd.notna(infer) and infer > 0 and n else np.nan})
latency_eff = pd.DataFrame(latency_rows).drop_duplicates(["model", "split", "inference_time_seconds"])
latency_eff.to_csv(TABLES / "latency_efficiency_summary.csv", index=False)

complexity_rows = []
for p in (MODELS / "baseline_model_artifacts").glob("*.joblib"):
    name = p.stem.replace("_classifier", "").replace("_regressor", "").replace("_", " ")
    complexity_rows.append({"model":name, "model_group":add_group_from_name(name), "number_of_parameters":np.nan, "number_of_estimators":np.nan, "model_size_mb":p.stat().st_size/(1024**2), "memory_peak_mb":np.nan})
for p in ARTIFACT_DIR.glob("*.joblib"):
    complexity_rows.append({"model":"B-TGPRF", "model_group":"Proposed", "number_of_parameters":len(final_components["residual_features"]), "number_of_estimators":np.nan, "model_size_mb":p.stat().st_size/(1024**2), "memory_peak_mb":np.nan})
model_complexity = pd.DataFrame(complexity_rows)
model_complexity.to_csv(TABLES / "model_complexity_summary.csv", index=False)

# Horizon sensitivity: h1 from B-TGPRF plus compact graph-temporal horizon-specific models for h6/h24 when targets exist.
horizon_rows = []
base_feature_cols = [c for c in feature_sets["full"] if c in model_meta.columns]
train_cols = ["split", "row_id"] + base_feature_cols + ["line_flow"] + [f"future_line_flow_h{h}" for h in [6,24] if f"future_line_flow_h{h}" in model_meta.columns] + [f"risk_binary_h{h}" for h in [6,24] if f"risk_binary_h{h}" in model_meta.columns]
train_source = pd.read_parquet(DATA_UNIFIED / "unified_powergrid_modeling_dataset.parquet", columns=list(dict.fromkeys(train_cols)))
train_source = train_source[train_source["split"].eq("train")].sample(min(60000, (train_source["split"].eq("train")).sum()), random_state=RANDOM_SEED)
for split, frame in frames.items():
    m = proposed_metrics[proposed_metrics["split"].eq(split)].iloc[0].to_dict()
    horizon_rows.append({"model":"B-TGPRF", "split":split, "horizon_hours":1, "MAE":m["MAE"], "RMSE":m["RMSE"], "Macro-F1":m["Macro-F1"], "Balanced Accuracy":m["Balanced Accuracy"], "Brier Score":m["Brier Score"], "Expected Calibration Error":m["Expected Calibration Error"]})
for h in [6, 24]:
    target = f"future_line_flow_h{h}"
    risk = f"risk_binary_h{h}"
    if target in train_source.columns and risk in train_source.columns:
        valid_train = train_source.dropna(subset=[target])
        if len(valid_train) > 1000:
            reg = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", HistGradientBoostingRegressor(max_iter=90, learning_rate=0.06, max_leaf_nodes=15, random_state=RANDOM_SEED))])
            reg.fit(valid_train[base_feature_cols], valid_train[target])
            valid_eval = frames["validation"].dropna(subset=[target]).copy()
            pred_valid_h = reg.predict(valid_eval[base_feature_cols])
            margin_valid = np.abs(pred_valid_h) - valid_eval["abs_flow_q95_train"].to_numpy(dtype=float)
            scale_h = np.median(np.abs(margin_valid - np.median(margin_valid))) * 1.4826
            scale_h = float(scale_h if np.isfinite(scale_h) and scale_h > 1e-9 else np.std(margin_valid) + 1e-9)
            prob_valid_h = 1 / (1 + np.exp(-(margin_valid / scale_h)))
            threshold_h = choose_threshold(valid_eval[risk].astype(int).to_numpy(), prob_valid_h)
            for split, frame in frames.items():
                eval_frame = frame.dropna(subset=[target]).copy()
                if len(eval_frame) == 0:
                    continue
                pred_h = reg.predict(eval_frame[base_feature_cols])
                margin_h = np.abs(pred_h) - eval_frame["abs_flow_q95_train"].to_numpy(dtype=float)
                prob_h = 1 / (1 + np.exp(-(margin_h / scale_h)))
                cm = classification_metrics(eval_frame[risk].astype(int).to_numpy(), prob_h, threshold_h)
                fm = forecasting_metrics(eval_frame[target].to_numpy(dtype=float), pred_h, eval_frame["line_flow"].to_numpy(dtype=float))
                horizon_rows.append({"model":"B-TGPRF compact horizon sensitivity", "split":split, "horizon_hours":h, "validation_threshold":threshold_h, "MAE":fm["MAE"], "RMSE":fm["RMSE"], "Macro-F1":cm["Macro-F1"], "Balanced Accuracy":cm["Balanced Accuracy"], "Brier Score":cm["Brier Score"], "Expected Calibration Error":cm["Expected Calibration Error"]})
        print("Horizon sensitivity computed:", h)
horizon_summary = pd.DataFrame(horizon_rows)
horizon_summary.to_csv(TABLES / "horizon_sensitivity_summary.csv", index=False)

display(error_reduction)
display(latency_eff.head(15))
display(model_complexity.head(15))
display(horizon_summary)
print("Saved comparison, latency, complexity, error-reduction, calibration, uncertainty, and horizon-sensitivity tables.")


Horizon sensitivity computed: 6


Horizon sensitivity computed: 24


,comparison,reference_model,split,reference_MAE,btgprf_MAE,relative_MAE_reduction,reference_RMSE,btgprf_RMSE
0,B-TGPRF vs validation-best baseline,ExtraTreesRegressor,validation,0.384038,0.349206,0.090700,0.627629,0.548273
1,B-TGPRF vs validation-best baseline,ExtraTreesRegressor,test,0.402188,0.410179,-0.019870,0.616060,0.600724
2,B-TGPRF vs validation-best baseline,ExtraTreesRegressor,external_test,0.374802,0.365028,0.026079,0.590289,0.547759
3,B-TGPRF vs validation-best SOTA,GRU,validation,1.323549,0.349206,0.736159,1.886771,0.548273
4,B-TGPRF vs validation-best SOTA,GRU,test,1.398934,0.410179,0.706792,2.002912,0.600724
5,B-TGPRF vs validation-best SOTA,GRU,external_test,1.292688,0.365028,0.717621,1.843251,0.547759


,model,model_group,split,training_time_seconds,inference_time_seconds,latency_per_sample_ms,throughput_samples_per_second
0,Persistence risk baseline,Persistence,validation,NaN,0.000267,0.000004,2.248919e+08
1,Persistence risk baseline,Persistence,test,NaN,0.000208,0.000003,2.885941e+08
2,Persistence risk baseline,Persistence,external_test,NaN,0.000156,0.000003,3.842046e+08
3,Seasonal previous-day risk baseline,Classical ML,validation,NaN,0.000303,0.000005,1.978412e+08
4,Seasonal previous-day risk baseline,Classical ML,test,NaN,0.000181,0.000003,3.315600e+08
5,Seasonal previous-day risk baseline,Classical ML,external_test,NaN,0.000207,0.000003,2.899240e+08
6,Bayesian Ridge risk approximation,Bayesian baseline,validation,NaN,0.061866,0.001031,9.698212e+05
7,Bayesian Ridge risk approximation,Bayesian baseline,test,NaN,0.056452,0.000941,1.062827e+06
8,Bayesian Ridge risk approximation,Bayesian baseline,external_test,NaN,0.053315,0.000889,1.125365e+06
9,Gaussian Naive Bayes,Bayesian baseline,validation,1.275247,0.076035,0.001267,7.890944e+05


,model,model_group,number_of_parameters,number_of_estimators,model_size_mb,memory_peak_mb
0,XGBoost Regressor,Boosting,NaN,NaN,0.234769,NaN
1,LightGBM,Boosting,NaN,NaN,0.541045,NaN
2,Extra Trees,Tree ensemble,NaN,NaN,49.296123,NaN
3,ElasticNet,Classical ML,NaN,NaN,0.005471,NaN
4,Logistic Regression,Classical ML,NaN,NaN,0.005732,NaN
5,HistGradientBoostingRegressor,Boosting,NaN,NaN,0.722727,NaN
6,Random Forest,Tree ensemble,NaN,NaN,15.377696,NaN
7,HistGradientBoostingClassifier,Boosting,NaN,NaN,0.653635,NaN
8,Ridge,Classical ML,NaN,NaN,0.005396,NaN
9,XGBoost,Boosting,NaN,NaN,0.196978,NaN


,model,split,horizon_hours,MAE,RMSE,Macro-F1,Balanced Accuracy,Brier Score,Expected Calibration Error,validation_threshold
0,B-TGPRF,validation,1,0.349206,0.548273,0.913552,0.882884,0.009371,0.004024,NaN
1,B-TGPRF,test,1,0.410179,0.600724,0.827807,0.762779,0.028011,0.029302,NaN
2,B-TGPRF,external_test,1,0.365028,0.547759,0.895981,0.874677,0.014719,0.006395,NaN
3,B-TGPRF compact horizon sensitivity,validation,6,3.578232,4.781061,0.550834,0.577733,0.048123,0.067141,0.238155
4,B-TGPRF compact horizon sensitivity,test,6,3.644728,4.802018,0.583275,0.601162,0.069005,0.037086,0.238155
5,B-TGPRF compact horizon sensitivity,external_test,6,3.608666,4.818461,0.561492,0.594079,0.055353,0.060893,0.238155
6,B-TGPRF compact horizon sensitivity,validation,24,3.030144,4.163216,0.678189,0.707897,0.051562,0.109586,0.386191
7,B-TGPRF compact horizon sensitivity,test,24,3.014748,4.163918,0.644751,0.630168,0.072135,0.084458,0.386191
8,B-TGPRF compact horizon sensitivity,external_test,24,3.039815,4.195347,0.656386,0.680231,0.058896,0.106482,0.386191


Saved comparison, latency, complexity, error-reduction, calibration, uncertainty, and horizon-sensitivity tables.


## Proposed Model Reports

These reports document the validation-only selection protocol, architecture, and limitations used by the final manuscript notebook.

In [8]:

selection_md = f"""# Proposed Model Selection Protocol

## Model Name
B-TGPRF: Bayesian Temporal Graph Probabilistic Risk Forecaster.

## Validation-Only Selection
All architecture choices are selected using validation data only. Test and strict external-test partitions are evaluated after the configuration is fixed and are not used for thresholds, calibration, component weights, or selection.

## Candidate Components
The search evaluates point-forecasting backbone choice, graph/topology/operating-regime feature representation, residual correction, calibration method, Bayesian posterior-style risk layer, and interval strategy.

## Composite Objective
The final configuration maximizes an equal-family validation score:
1. Safety/risk detection: Macro-F1, Balanced Accuracy, Recall, and false-negative reduction.
2. Forecasting accuracy: MAE, RMSE, R2, and normalized RMSE.
3. Calibration and uncertainty: Brier Score, Expected Calibration Error, pinball loss, and interval calibration error.
4. Efficiency: fit time and inference time.

Equal family weighting avoids arbitrary post-hoc emphasis after observing test or external-test metrics.

## Selected Configuration
{final_configuration.to_csv(index=False)}

## Forecast Component Weights
{forecast_weight_table.to_csv(index=False)}
"""
(REPORTS / "proposed_model_selection_protocol.md").write_text(selection_md)

arch_md = f"""# B-TGPRF Architecture Description

B-TGPRF is a hybrid Bayesian temporal graph probabilistic model for transmission-grid overload risk forecasting.

## Architecture Components
1. Validation-selected point-forecasting backbone: `{final_spec['backbone']}`.
2. Temporal lag and rolling representations from the unified modelling dataset.
3. Graph-neighborhood aggregation features for adjacent-line flow context.
4. Topology-aware line features including endpoint degree, line centrality proxy, and available capacity-derived ratios.
5. Operating-regime awareness for high-load, high-generation, high-ramp, low-margin, and stress regimes.
6. Bayesian posterior-style risk layer using Bayesian Ridge and Gaussian posterior evidence over risk-design features.
7. Validation-only probability calibration: `{final_spec['calibration']}`.
8. Residual correction for near-threshold and high-risk regimes: `{final_spec['residual_strategy']}`.
9. Probabilistic interval estimation: `{final_spec['interval_strategy']}`.
10. Validation-derived Normal, Warning, and Critical risk decision thresholds.

## Outputs
Prediction files contain point line-flow forecast, lower interval, median forecast, upper interval, interval width, overload risk probability, calibrated overload risk probability, risk class, uncertainty score, and external-test prediction rows.
"""
(REPORTS / "proposed_model_architecture_description.md").write_text(arch_md)

limits_md = """# Proposed Model Limitations

1. The executable protocol uses the available matched 2020 suffix groups in the current project state.
2. The modelling dataset uses a selected critical-line protocol for computational feasibility.
3. The Bayesian risk layer is a feasible posterior-style approximation over graph-temporal risk features, not a full physical Bayesian power-flow simulator.
4. Validation data are used for architecture selection, calibration, and decision thresholds; test and strict external-test metrics are preserved for final evaluation only.
5. Any baseline or SOTA model that outperforms B-TGPRF on a specific metric must be reported transparently.
"""
(REPORTS / "proposed_model_limitations.md").write_text(limits_md)

print("Saved reports:")
for p in ["proposed_model_selection_protocol.md", "proposed_model_architecture_description.md", "proposed_model_limitations.md"]:
    print(REPORTS / p)


Saved reports:
/Users/talgatazykanov/Desktop/Science works/Bayiessian_Asem_Almaty_2026_05/Bayesian_PowerGrid_Risk_Forecasting_N1/Reports/proposed_model_selection_protocol.md
/Users/talgatazykanov/Desktop/Science works/Bayiessian_Asem_Almaty_2026_05/Bayesian_PowerGrid_Risk_Forecasting_N1/Reports/proposed_model_architecture_description.md
/Users/talgatazykanov/Desktop/Science works/Bayiessian_Asem_Almaty_2026_05/Bayesian_PowerGrid_Risk_Forecasting_N1/Reports/proposed_model_limitations.md
